In [6]:
import bz2

file_path = "/Users/a1/github_reps/butterboard_CRAG/data/crag_task_1_and_2_dev_v4.jsonl.bz2"

with bz2.open(file_path, "rt", encoding="utf-8") as f:
    print(f.readline(228))  # Вывести первую строку файла

{"interaction_id": "7bb29eb4-12f9-45f9-bf8a-66832b3c8962", "query_time": "03/10/2024, 23:19:21 PT", "domain": "sports", "question_type": "post-processing", "static_or_dynamic": "static", "query": "how many 3-point attempts did s


# Filtering the long-tailed questions

In [10]:
import bz2
import json
import pandas as pd
from collections import Counter

CRAG_DATA_PATH = "data/crag_task_1_and_2_dev_v4.jsonl.bz2"
FILTERED_DATA_PATH = "data/filtered_long_tailed_questions_test.jsonl"

# Define long-tailed question criteria
RARE_THRESHOLD = 5  # Words appearing less than 5 times
LONG_TAIL_TYPES = ["multi-hop", "comparison", "false_premise"]  # Harder question types
SAMPLE_SIZE = 10  # Limit dataset size

def load_crag_data(filepath):
    """Load CRAG dataset from bz2 compressed JSONL format."""
    with bz2.open(filepath, "rt") as f:
        return [json.loads(line) for line in f]

def filter_long_tailed_questions(data):
    """Extract long-tailed questions using word frequency and question type."""
    df = pd.DataFrame(data)

    # Compute word frequencies
    all_words = " ".join(df["query"]).split()
    word_counts = Counter(all_words)

    def is_long_tailed(query, question_type):
        words = query.split()
        rare_words = [word for word in words if word_counts[word] < RARE_THRESHOLD]
        return len(rare_words) > 2 or question_type in LONG_TAIL_TYPES

    filtered_df = df[df.apply(lambda row: is_long_tailed(row["query"], row["question_type"]), axis=1)]
    return filtered_df.sample(n=min(SAMPLE_SIZE, len(filtered_df)), random_state=42)

if __name__ == "__main__":
    crag_data = load_crag_data(CRAG_DATA_PATH)
    filtered_data = filter_long_tailed_questions(crag_data)
    
    # Save filtered dataset
    with open(FILTERED_DATA_PATH, "w") as f:
        for record in filtered_data.to_dict(orient="records"):
            f.write(json.dumps(record) + "\n")

    print(f"Filtered {len(filtered_data)} long-tailed questions saved to {FILTERED_DATA_PATH}")


Filtered 10 long-tailed questions saved to data/filtered_long_tailed_questions_test.jsonl


# Creating limited balanced dataset

In [12]:
import bz2
import json
import pandas as pd
from collections import Counter

CRAG_DATA_PATH = "data/crag_task_1_and_2_dev_v4.jsonl.bz2"
FILTERED_DATA_PATH = "data/balanced_100_questions.jsonl"
SAMPLE_PER_CATEGORY = 10  # Number of questions per category

def load_crag_data(filepath):
    """Load CRAG dataset from bz2 compressed JSONL format."""
    with bz2.open(filepath, "rt") as f:
        return [json.loads(line) for line in f]

def sample_balanced_questions(data):
    """Select an equal number of questions from each category."""
    df = pd.DataFrame(data)
    categories = df["question_type"].unique()
    
    sampled_dfs = []
    for category in categories:
        category_df = df[df["question_type"] == category]
        sampled_dfs.append(category_df.sample(n=min(SAMPLE_PER_CATEGORY, len(category_df)), random_state=42))
    
    return pd.concat(sampled_dfs)

if __name__ == "__main__":
    crag_data = load_crag_data(CRAG_DATA_PATH)
    balanced_data = sample_balanced_questions(crag_data)
    
    # Save balanced dataset
    with open(FILTERED_DATA_PATH, "w") as f:
        for record in balanced_data.to_dict(orient="records"):
            f.write(json.dumps(record) + "\n")

    print(f"Balanced dataset with {len(balanced_data)} questions saved to {FILTERED_DATA_PATH}")


Balanced dataset with 80 questions saved to data/balanced_100_questions.jsonl


# Compressing files to bz2

In [1]:
import bz2
import json

def compress_jsonl_to_bz2(jsonl_path, bz2_path):
    with open(jsonl_path, 'rt') as jsonl_file, bz2.open(bz2_path, 'wt') as bz2_file:
        for line in jsonl_file:
            bz2_file.write(line)

if __name__ == "__main__":
    jsonl_path = 'data/russian_crag.jsonl'  # Replace with the path to your JSONL file
    bz2_path = 'data/russian_crag_test.jsonl.bz2'  # Replace with the desired output path

    compress_jsonl_to_bz2(jsonl_path, bz2_path)
    print(f"Compressed {jsonl_path} to {bz2_path}")

Compressed data/russian_crag.jsonl to data/russian_crag_test.jsonl.bz2


In [13]:
import bz2
import json

def compress_jsonl_to_bz2(jsonl_path, bz2_path):
    with open(jsonl_path, 'rt') as jsonl_file, bz2.open(bz2_path, 'wt') as bz2_file:
        for line in jsonl_file:
            bz2_file.write(line)

if __name__ == "__main__":
    jsonl_path = 'data/balanced_100_questions.jsonl'  # Replace with the path to your JSONL file
    bz2_path = 'data/balanced_100_questions.jsonl.bz2'  # Replace with the desired output path

    compress_jsonl_to_bz2(jsonl_path, bz2_path)
    print(f"Compressed {jsonl_path} to {bz2_path}")

Compressed data/balanced_100_questions.jsonl to data/balanced_100_questions.jsonl.bz2


## Connection test

In [41]:
import httpx
import json
from openai import OpenAI
from dotenv import dotenv_values

config = dotenv_values('.env')

In [2]:
import certifi

certifi.where()

'/Users/a1/github_reps/butterboard_CRAG/venv/lib/python3.11/site-packages/certifi/cacert.pem'

In [3]:
import requests
requests.get('https://ya.ru')

<Response [200]>

In [5]:
httpx.get('https://ya.ru', follow_redirects=True)

<Response [200 Ok]>

In [6]:
GIGACHAT_API_KEY = config["GIGACHAT_API_KEY"]
GIGACHAT_API_URL = 'https://gigachat.devices.sberbank.ru/api/v1'
CA_CERT_PATH = "cert/Russian Trusted Root CA.pem" 

http_client = httpx.Client(verify=CA_CERT_PATH)
http_client_base = httpx.Client()

http_client_base.get(
    url = 'https://ya.ru'
)

<Response [302 Moved temporarily]>

In [50]:
# GigaChat API Configuration
GIGACHAT_API_KEY = config["GIGACHAT_API_KEY"]
GIGACHAT_API_URL = 'https://gigachat.devices.sberbank.ru/api/v1'
CA_CERT_PATH = "cert/russiantrustedca.pem" 

import requests

url = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth"

payload = 'scope=GIGACHAT_API_PERS'
headers = {
  'RqUID': '6f0b1291-c7f3-43c6-bb2e-9f3efb2dc98e',
  'Content-Type': 'application/x-www-form-urlencoded',
  'Authorization': f'Bearer {GIGACHAT_API_KEY}'
}

response = requests.request("POST", url, headers=headers, data=payload, verify=False)

sber_api_response = response.json()



/Users/a1/github_reps/butterboard_CRAG/venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ngw.devices.sberbank.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [54]:
sber_api_response

{'access_token': 'eyJjdHkiOiJqd3QiLCJlbmMiOiJBMjU2Q0JDLUhTNTEyIiwiYWxnIjoiUlNBLU9BRVAtMjU2In0.UDITTaGScrsXKbNlfW2KQejjZEnpudopE93Dv1brriYHVKkG5GNsFi6NNK5632gFWCISvTph5vaXtXRJJzt_DWlycM3PSwqDxuUq1KwXgk-lEFQgBSbCgMDydKqelpf_wN7obfNutLRmVRKZ3EEYk_3-pQuaC4eXaYehBiNnowkBRXONL9Lox-acAsdDZusDdEK2xoJUX8BHs9HX_pJiRj-fwLErm3vEqAdYLzrXdTT2ktSGJHfNhaKwVfpcNpsDmRR6ivhSCtmx1xN52UMAwK0HWuHHjnSg1KDfWeVX7z_yJcaL6dlJkWxU67WYiKHDmyuc2dwZnX5lXLO269rtoQ.G9gfWivf7-Lxy8p3wJ-oJQ.XQU-7yTkjA_RggYbwjlmRri0g7QV7ykBd70NEhE1xVI0-pTyEP9rglmrZiIxz37-4zjYvqPGTxbfD1EK9M9vVeicXnr6qNlockwZbPdlbIsSAku_sZXNGa9HLFSBHKWg3SKyLp02V7v5XnKBoF58PUMHg3rR1gFcBjg2GSOmyHOTwtDbdhVrR8y0SgCmYYgAnKeIcX07PitD2DUqJujwoEEtamRZGUnoQBxVoH0MBkH-tpxkyQCVv9seS1ZE37fzrUkLs2Z6ykrAhve9I8xlbhn5XYs9ZDkYnm845iWeiswNCqSW6idJzzZBT-SqlygVpE6uiOxCXbR3qTIQIvUHj7kC4661cAYnvE86aL7BEh88hbSS-BndOE_7hPvhkqUknNJaO9zekfQuDks4A7rBJmHvW28u7VU1AMj48LwCpRCOV3COQ5GW1_25b7NtH8ZnI7eR9Q3Nq3hQG5YPsuDc6sgZJzHh4c0uBiv84ytIoIwsprbVIYY4Qen7vqejZPeaUsEspiL3JVttI7wZUbXvNQfOhHXV

In [51]:
sber_api_key = sber_api_response.get("access_token", "Unknown")

In [52]:
http_client = httpx.Client(verify=False)
client = OpenAI(api_key=sber_api_key, base_url=GIGACHAT_API_URL, http_client=http_client)


completion = client.chat.completions.create(
    model="GigaChat",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "Hello World!"
        }
    ]
)

In [53]:
completion

ChatCompletion(id=None, choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))], created=1741455448, model='GigaChat:1.0.26.20', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=10, prompt_tokens=22, total_tokens=32, completion_tokens_details=None, prompt_tokens_details=None, precached_prompt_tokens=0))

In [1]:
import json

file_path = "/Users/a1/github_reps/butterboard_CRAG/data/filtered_long_tailed_questions_test.jsonl"  # Replace with your file path

with open(file_path, 'r', encoding='utf-8') as file:
    first_line = next(file, None)  # Get first line (or None if empty)
    if first_line:
        first_record = json.loads(first_line.strip())
    else:
        first_record = None  # File is empty

# Now `first_record` contains the first JSON object (or None)
print(first_record)

{'interaction_id': '2ab5517d-2646-4375-a4b7-3f15ccc4cfd4', 'query_time': '03/13/2024, 10:04:31 PT', 'domain': 'music', 'question_type': 'false_premise', 'static_or_dynamic': 'static', 'query': 'when was alex van halen the drummer for the band santana?', 'answer': 'invalid question', 'search_results': [{'page_name': 'Where is Alex Van Halen, the legendary recluse drummer', 'page_url': 'https://rockandrollgarage.com/where-is-alex-van-halen-the-recluse-drummer/', 'page_snippet': 'Alex Van Halen is the legendary drummer that kept Van Halen&#x27;s rythm throughout all the band&#x27;s career. He has always been the band&#x27;s quiet one beside his brother Eddie Van Halen. On the David Lee Roth era, the singer was the most crazy and communicative one, the same in the Sammy Hagar era.Alex Van Halen is the legendary drummer that kept Van Halen‘s rhythm. He has always been the band’s quiet one alongside his brother Eddie Van Halen. Advertisement The band doesn’t perform live since 2015 and this 

In [3]:
from models.rag_yandexgpt_baseline import RAGModel
import uuid
from datetime import datetime

rag_model = RAGModel()

''' 
      batch_interaction_ids = batch["interaction_id"]
        queries = batch["query"]
        batch_search_results = batch["search_results"]
        query_times = batch["query_time"]

'''

rag_model.batch_generate_answer(
    batch= {
            "interaction_id": [first_record['interaction_id']],
            "query": [first_record['query']],
            "search_results": [first_record['search_results']],
            "query_time": [first_record['query_time']],
            "answer": [first_record['answer']],
        }
    )

OrderedDict([('YCLOUD_API_TOKEN', 'AQVN35-136OmO2i6qctSl6uWJe7F7812_2qKyeJi'), ('YCLOUD_FOLDER_ID', 'b1gmo0oc6ga63led7a7u'), ('YANDEX_MODEL', 'yandexgpt'), ('GIGACHAT_API_KEY', 'OWMyMjZmY2EtZDQwZS00ZmRlLThiM2MtNzY4ZWUzZTA5YjAyOjMzNjIwNTU1LWUyMmQtNDZhMC1hYTQ4LTUxMWYwZDc5MGUwZg=='), ('GIGACHAT_MODEL', 'GigaChat'), ('OPENAI_API_KEY', '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'), ('RU_PROXY_OPENAI_API_KEY', 'sk-Oo7PYqAzfAxW5VEHtbBvJR6ho1Lsapp8'), ('OPENAI_MODEL', 'deepseek/deepseek-chat-v3-0324:free'), ('FILES_PATH', 'tests/data/'), ('TOKENIZERS_PARALLELISM', 'false')])
Extract chunks. Enter for loop
Extract chunks. Exit for loop
Flatten chunks. Enter for loop
Flatten chunks. Exit for loop
Getting embeddings from Yandex
Getting embeddings from Yandex


["I don't know."]

In [3]:
first_record['answer']

'invalid question'

In [3]:
from yandex_chain import YandexLLM, YandexEmbeddings, __version__
import langchain
from dotenv import dotenv_values

config = dotenv_values('.env')

print(f"Using yandex_chain version {__version__}, langchain=={langchain.__version__}")
gpt = YandexLLM(api_key=config['YCLOUD_API_TOKEN'], folder_id=config['YCLOUD_FOLDER_ID'])
print(gpt.invoke('Привет! Придумай 10 новых слов для приветствия.'))
print(f"Usage: {gpt.totalTokens} tokens")

emb = YandexEmbeddings(api_key=config['YCLOUD_API_TOKEN'], folder_id=config['YCLOUD_FOLDER_ID'])
print(emb.embed_document('Hello, world'))

Using yandex_chain version 0.0.10, langchain==0.2.1
1. Приветствую!
2. Здраствуй!
3. Салют!
4. Наше вам привет!
5. Приветствие моё!
6. Рад тебя приветствовать!
7. Приветствуйте!
8. Здорово!
9. Здравия желаю!
10. Приветственный поклон!
Usage: 90 tokens
[0.050079345703125, 0.00968170166015625, -0.00274658203125, -0.0692138671875, 0.004512786865234375, 0.055023193359375, -0.0146331787109375, -0.081787109375, 0.011932373046875, -0.057525634765625, -0.031890869140625, -0.0694580078125, -0.0026836395263671875, -0.046722412109375, -0.0147247314453125, 0.0266265869140625, 0.060546875, -0.013092041015625, -0.0201416015625, -0.07061767578125, 0.07666015625, -0.033935546875, -0.0567626953125, 0.00481414794921875, 0.0258331298828125, 0.00875091552734375, -0.08355712890625, -0.018310546875, 0.07281494140625, 0.016265869140625, -0.16162109375, 0.053070068359375, -0.01276397705078125, 0.051116943359375, -0.0460205078125, -0.045501708984375, -0.01251983642578125, -0.04193115234375, 0.11663818359375, 0

In [23]:
from openai import OpenAI
from dotenv import dotenv_values

config = dotenv_values('.env')

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=config['OPENAI_API_KEY'],
)

completion = client.chat.completions.create(
  extra_headers={
    "HTTP-Referer": "<YOUR_SITE_URL>", # Optional. Site URL for rankings on openrouter.ai.
    "X-Title": "<YOUR_SITE_NAME>", # Optional. Site title for rankings on openrouter.ai.
  },
  model=config['OPENAI_MODEL'],
  messages=[
    {
      "role": "user",
      "content": "What is the meaning of life?"
    }
  ]
)

print(completion.choices[0].message.content)


The meaning of life is a complex and deeply philosophical question that varies widely based on individual beliefs, cultural backgrounds, and personal experiences. For some, it may involve seeking happiness, fulfilling relationships, or making a positive impact on the world. Others may find meaning through spiritual beliefs, personal growth, or the pursuit of knowledge.

Philosophers have offered different perspectives: existentialists might argue that individuals must create their own meaning in a seemingly indifferent universe, while religious traditions might provide a framework for understanding life’s purpose based on divine plans or moral principles. Ultimately, the meaning of life is subjective, and each person may find their own answer through exploration, reflection, and experiences.


In [6]:
import numpy as np

np.zeros(256)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

# Создание датасета

In [29]:
import os
import json
import uuid
from datetime import datetime
from openai import OpenAI
from faker import Faker
from dotenv import dotenv_values

# Load configuration
config = dotenv_values('.env')

# Initialize OpenAI client for OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=config['OPENAI_API_KEY'],
)

# Initialize Faker for Russian data
fake = Faker('ru_RU')

# Dataset configuration
DOMAINS = [
    "музыка", "кино", "технологии", "спорт", 
    "история", "наука", "география", "литература"
]

QUESTION_TYPES = [
    "длинный_хвост", "после_обработки", "ложная_предпосылка",
    "динамический_факт", "сравнение"
]

def generate_russian_question(domain):
    prompt = f"""
    Сгенерируй сложный вопрос на русском языке на тему: {domain}.
    Вопрос должен быть типа 'длинный хвост' - конкретный и редко задаваемый.
    Примеры хороших вопросов:
    - Каков был средний бюджет всех фильмов серии 'Гарри Поттер' с поправкой на инфляцию?
    - Сколько профессиональных матчей проиграл Фёдор Емельяненко до завершения карьеры?
    - Какие российские учёные участвовали в разработке стандарта Bluetooth 5.2?
    
    Верни только сам вопрос без кавычек и дополнительного текста.
    """
    
    completion = client.chat.completions.create(
        extra_headers={
            "HTTP-Referer": "https://github.com/your-repo",  # Update with your info
            "X-Title": "Russian CRAG Dataset Generator"
        },
        model=config['OPENAI_MODEL'],
        messages=[
            {"role": "system", "content": "You are a helpful assistant that generates complex Russian questions."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )
    
    return completion.choices[0].message.content.strip()

def generate_search_results(question):
    return [{
        "page_name": fake.sentence(),
        "page_url": fake.url(),
        "page_snippet": ' '.join(fake.paragraphs(nb=3)),
        "page_result": f"<html>{' '.join(fake.paragraphs(nb=5))}</html>"
    } for _ in range(3)]

def create_dataset(num_questions=1):
    dataset = []
    
    for i in range(num_questions):
        
        question = generate_russian_question(domain=DOMAINS[0])
        
        entry = {
            "interaction_id": str(uuid.uuid4()),
            "query_time": datetime.now().strftime("%m/%d/%Y, %H:%M:%S PT"),
            "domain": fake.random_element(DOMAINS),
            "question_type": fake.random_element(QUESTION_TYPES),
            "static_or_dynamic": fake.random_element(["static", "dynamic"]),
            "query": question,
            "answer": "неизвестно",
            "search_results": generate_search_results(question)
        }
        
        dataset.append(entry)
        print(f"Сгенерирован вопрос {i+1}/{num_questions}")
    
    # Save to JSONL
    with open("russian_crag_dataset.jsonl", "w", encoding="utf-8") as f:
        for item in dataset:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    
    print("Датасет успешно создан!")

if __name__ == "__main__":
    create_dataset(1)

Сгенерирован вопрос 1/1
Датасет успешно создан!


In [31]:
import uuid
import json
from datetime import datetime
from IPython.display import display, clear_output
import ipywidgets as widgets

# Widget styles
style = {'description_width': '150px'}
layout = widgets.Layout(width='80%')

class DatasetCreator:
    def __init__(self):
        self.entries = []
        self.current_entry = {}
        self.create_widgets()
        
    def create_widgets(self):
        # Core fields
        self.domain = widgets.Dropdown(
            options=['finance', 'music', 'movie', 'sports', 'open'],
            description='Domain:',
            style=style,
            layout=layout
        )
        
        self.question_type = widgets.Dropdown(
            options=['simple', 'simple_w_condition', 'comparison', 'aggregation',
                     'set', 'false_premise', 'post-processing', 'multi-hop'],
            description='Question Type:',
            style=style,
            layout=layout
        )
        
        self.static_dynamic = widgets.Dropdown(
            options=['static', 'slow-changing', 'fast-changing', 'real-time'],
            description='Static/Dynamic:',
            style=style,
            layout=layout
        )
        
        self.query = widgets.Textarea(
            description='Question:',
            style=style,
            layout=layout,
            rows=3
        )
        
        self.answer = widgets.Textarea(
            description='Correct Answer:',
            style=style,
            layout=layout,
            rows=3
        )
        
        # Alternative Answers
        self.alt_answers = widgets.SelectMultiple(
            description='Alternative Answers:',
            options=[],
            style=style,
            layout=widgets.Layout(width='80%', height='100px')
        )
        self.new_alt_answer = widgets.Text(
            description='New Alt Answer:',
            style=style,
            layout=layout
        )
        self.add_alt_answer = widgets.Button(description='Add Alt Answer')
        self.add_alt_answer.on_click(self.add_alt_answer_handler)
        
        # Search Results
        self.search_results = widgets.Accordion(children=[])
        self.add_search_result = widgets.Button(description='Add Search Result')
        self.add_search_result.on_click(self.add_search_result_handler)
        
        # Split selection
        self.split = widgets.ToggleButtons(
            options=[('Validation', 0), ('Test', 1)],
            description='Split:',
            style=style
        )
        
        # Control buttons
        self.submit = widgets.Button(description='Submit Entry', button_style='success')
        self.submit.on_click(self.submit_handler)
        self.clear = widgets.Button(description='Clear Form', button_style='warning')
        self.clear.on_click(self.clear_form)
        
        # Assemble UI
        self.form = widgets.VBox([
            self.domain,
            self.question_type,
            self.static_dynamic,
            self.query,
            self.answer,
            widgets.HBox([self.new_alt_answer, self.add_alt_answer]),
            self.alt_answers,
            widgets.Label('Search Results:'),
            self.add_search_result,
            self.search_results,
            self.split,
            widgets.HBox([self.submit, self.clear])
        ])
        
    def add_alt_answer_handler(self, b):
        if self.new_alt_answer.value:
            new_options = list(self.alt_answers.options) + [self.new_alt_answer.value]
            self.alt_answers.options = new_options
            self.new_alt_answer.value = ''
            
    def add_search_result_handler(self, b):
        result_widgets = [
            widgets.Text(description='Page Title:', style=style, layout=layout),
            widgets.Text(description='URL:', style=style, layout=layout),
            widgets.Textarea(description='Snippet:', style=style, layout=layout, rows=3),
            widgets.Textarea(description='HTML:', style=style, layout=layout, rows=5),
            widgets.Text(description='Last Modified:', style=style, layout=layout)
        ]
        
        new_result = widgets.Accordion(children=[widgets.VBox(result_widgets)])
        new_result.set_title(0, f'Search Result {len(self.search_results.children)+1}')
        
        self.search_results.children = list(self.search_results.children) + [new_result]
        
    def clear_form(self, b):
        for widget in self.form.children:
            if isinstance(widget, (widgets.Dropdown, widgets.Text, widgets.Textarea)):
                widget.value = ''
            elif isinstance(widget, widgets.SelectMultiple):
                widget.options = []
        self.search_results.children = []
        
    def submit_handler(self, b):
        entry = {
            "interaction_id": str(uuid.uuid4()),
            "query_time": datetime.now().strftime("%m/%d/%Y, %H:%M:%S PT"),
            "domain": self.domain.value,
            "question_type": self.question_type.value,
            "static_or_dynamic": self.static_dynamic.value,
            "query": self.query.value,
            "answer": self.answer.value,
            "alt_ans": list(self.alt_answers.options),
            "split": self.split.value,
            "search_results": []
        }
        
        # Process search results
        for result in self.search_results.children:
            children = result.children[0].children
            entry["search_results"].append({
                "page_name": children[0].value,
                "page_url": children[1].value,
                "page_snippet": children[2].value,
                "page_result": children[3].value,
                "page_last_modified": children[4].value
            })
        
        self.entries.append(entry)
        self.save_to_file()
        self.clear_form(None)
        print("Entry saved successfully!")
        
    def save_to_file(self):
        with open("dataset.jsonl", "a", encoding="utf-8") as f:
            json.dump(self.entries[-1], f, ensure_ascii=False)
            f.write("\n")
            
    def show(self):
        display(self.form)

# Run the UI
creator = DatasetCreator()
creator.show()

In [39]:
import yandex_search

In [40]:
config = dotenv_values('.env')

ys = yandex_search.Yandex(api_user=config['YCLOUD_FOLDER_ID'], api_key=config['YCLOUD_SEARCH_TOKEN'])

In [41]:
ys.search('яндекс')

ConfigException: Invalid key version 'AQVN2_VbD0g4GzKq6MPVSuT1Kx7aLoUKHykgs6_N'

# Creating Russian jsonl dataset

In [9]:
import json

def append_to_jsonl(data, filename):
    """Append a json object to a jsonl file."""
    with open(filename, 'a', encoding='utf-8') as f:
        json_str = json.dumps(data)
        f.write(json_str + '\n')

# Example usage
new_data = {
  "interaction_id": "p5q6r7s8-t9u0-v1w2-x3y4-z5a6b7c8d9e0",
  "query_time": "2025-03-27T18:45:00+03:00",
  "domain": "finance",
  "question_type": "multi-hop",
  "static_or_dynamic": "fast-changing",
  "query": "Как изменился курс рубля к доллару и евро за последний месяц, и какие факторы оказали наибольшее влияние на его динамику?",
  "answer": "За последний месяц курс рубля укрепился: к доллару на 3.2% (с 98.5 до 95.3 руб/$), к евро на 2.8% (с 106.7 до 103.7 руб/€). Основные факторы влияния: повышение цен на нефть (+5.7% до $89.3 за баррель Brent), увеличение экспорта IT-услуг (+12% за квартал), и ужесточение валютного контроля ЦБ РФ. Дополнительно поддержку оказало снижение ключевой ставки ФРС США на 0.25 п.п.",
  "alt_ans": [
    "Рубль укрепился: -3.2% к $ (до 95.3), -2.8% к € (до 103.7). Причины: рост цен на нефть, экспорт IT, меры ЦБ РФ",
    "Укрепление рубля на ~3% к $ и €. Факторы: нефть +5.7%, IT-экспорт +12%, меры ЦБ РФ, снижение ставки ФРС"
  ],
  "split": 1,
  "search_results": [
    {
      "page_name": "Центральный банк РФ - Динамика курсов валют",
      "page_url": "https://www.cbr.ru/currency_base/dynamics/",
      "page_snippet": "Официальные курсы валют ЦБ РФ на заданную дату и за период...",
      "page_result": "<html><body><table class='currency-rates'><tr><th>Дата</th><th>USD</th><th>EUR</th></tr><tr><td>27.02.2025</td><td>98.5043</td><td>106.7231</td></tr><tr><td>27.03.2025</td><td>95.3176</td><td>103.7154</td></tr></table></body></html>",
      "page_last_modified": "2025-03-27"
    },
    {
      "page_name": "Московская биржа - Итоги торгов",
      "page_url": "https://www.moex.com/ru/marketdata/",
      "page_snippet": "Итоги торгов и динамика курсов валют на Московской бирже...",
      "page_result": "<html><body><div class='trading-results'><h3>Изменение курса за месяц</h3><p>USD/RUB: -3.2%</p><p>EUR/RUB: -2.8%</p></div></body></html>",
      "page_last_modified": "2025-03-27"
    },
    {
      "page_name": "Министерство финансов РФ - Цены на нефть",
      "page_url": "https://minfin.gov.ru/ru/document/?id_4=300143",
      "page_snippet": "Динамика цен на нефть марки Urals и Brent...",
      "page_result": "<html><body><div class='oil-prices'><h4>Цена на нефть Brent</h4><p>Текущая: $89.3 за баррель</p><p>Изменение за месяц: +5.7%</p></div></body></html>",
      "page_last_modified": "2025-03-26"
    },
    {
      "page_name": "РБК - Экспорт IT-услуг",
      "page_url": "https://www.rbc.ru/technology_and_media/27/03/2025/60af3a9a9a7947584daf6921",
      "page_snippet": "Анализ экспорта IT-услуг из России за первый квартал 2025 года...",
      "page_result": "<html><body><blockquote>Экспорт IT-услуг из России в первом квартале 2025 года вырос на 12% по сравнению с аналогичным периодом прошлого года, достигнув $3.2 млрд.</blockquote></body></html>",
      "page_last_modified": "2025-03-27"
    },
    {
      "page_name": "Федеральная резервная система США - Решения по ставке",
      "page_url": "https://www.federalreserve.gov/newsevents/pressreleases/monetary20250319a.htm",
      "page_snippet": "Федеральный комитет по открытым рынкам принял решение по ключевой ставке...",
      "page_result": "<html><body><div class='fed-decision'><h3>Решение ФРС США</h3><p>Ключевая ставка снижена на 0.25 процентных пункта</p><p>Новый диапазон: 4.75% - 5.00%</p></div></body></html>",
      "page_last_modified": "2025-03-19"
    }
  ]
}





append_to_jsonl(new_data, 'data/russian_crag.jsonl')